# AsyncUA factory clients: publisher & subscriber for `M01.Temperature`

This notebook is designed to work **with your existing Async Factory OPC UA server** from
`OPC_Server_template.ipynb`.

It implements two separate clients:

1. **Publisher client**  
   - connects to your running server  
   - finds the object `M01`  
   - periodically writes new values to the variable `Temperature`

2. **Subscriber client**  
   - connects to the same server  
   - subscribes to `M01.Temperature`  
   - reacts to data changes via a callback

> **Prerequisite:** Your server from `OPC_Server_template.ipynb` must already be running  
> (i.e. `FactoryOpcUaServer().run()` is active).

## 1. Install required packages

If `asyncua` and `nest_asyncio` are not installed yet, run the following cell once.


In [ ]:
!pip install -U asyncua nest_asyncio

## 2. Imports and event loop setup for Jupyter

Jupyter already runs an event loop. With `nest_asyncio` we can still use `await` in notebook cells
comfortably without running into loop conflicts.


In [ ]:
import asyncio
import math
from datetime import datetime

import nest_asyncio
nest_asyncio.apply()

from asyncua import ua, Client

# Configuration – these values must match your server notebook
SERVER_URL = "opc.tcp://localhost:4840/freeopcua/server/"
FACTORY_NS_URI = "http://ostfalia.de/ipt/factory"

print("Event loop setup complete. Server endpoint:", SERVER_URL)

## 3. Helper function: locate the `M01.Temperature` node

We assume that your server has created the following structure (as in the template notebook):

- `Objects`  
  - `Factory`  
    - `Machines`  
      - `M01`  
        - `Temperature`

The helper below encapsulates resolving this browse path so both clients can reuse it.


In [ ]:
async def get_machine_temperature_node(client: Client, machine: str = "M01"):
    """Return the Node for Factory.Machines.M01.Temperature."""
    # Determine namespace index from URI
    nsidx = await client.get_namespace_index(FACTORY_NS_URI)
    print(f"[CLIENT] Namespace index for {FACTORY_NS_URI!r}: {nsidx}")

    objects = client.nodes.objects
    factory = await objects.get_child([f"{nsidx}:Factory"])           # Objects/Factory
    machines_folder = await factory.get_child([f"{nsidx}:Machines"])  # Factory/Machines
    mXY = await machines_folder.get_child([f"{nsidx}:{machine}"])     # Machines/"machine"
    temp_node = await mXY.get_child([f"{nsidx}:Temperature"])         # "machine"/Temperature

    print(f"[CLIENT] Resolved temperature NodeId: {temp_node.nodeid}")
    return temp_node

## 4. Publisher client

The publisher client connects to the server and periodically writes new values
to `M01.Temperature`. Here we use a simple sine wave on top of a base temperature.


In [ ]:
async def publisher_client(runtime_seconds: float = 20.0, interval: float = 1.0):
    """Periodically write new temperature values to M01.Temperature."""
    async with Client(url=SERVER_URL) as client:
        print("[PUBLISHER] Connected to server:", SERVER_URL)
        temp_node = await get_machine_temperature_node(client, "M01")

        loop = asyncio.get_running_loop()
        start = loop.time()
        t = 0.0
        while True:
            now = loop.time()
            if now - start > runtime_seconds:
                break

            # Example: temperature as (25 + 5 * sin(t))
            value = 25.0 + 5.0 * math.sin(t)
            await temp_node.write_value(ua.Variant(value, ua.VariantType.Double))
            ts = datetime.now().strftime("%H:%M:%S")
            print(f"[PUBLISHER {ts}] Written temperature: {value:.2f} °C")

            t += interval
            await asyncio.sleep(interval)

        print("[PUBLISHER] Finished.")

## 5. Subscriber client

The subscriber client:

- connects to the server,
- resolves `M01.Temperature`,
- creates a subscription,
- registers a handler that is called whenever the value changes.

### Why does the subscriber have a `publishing_interval_ms`?

In OPC UA, a **Subscription** has a **publishing interval**. This is **not** the interval at which
the server-side logic updates the value, and it is also not the publisher client's interval.

Instead, it tells the server:

> *“Please check for changes and send publish responses roughly every X milliseconds.”*

So:

- The **publisher client** controls how often values are written (e.g. every 1 second).
- The **subscription**’s `publishing_interval_ms` controls how often the client expects updates
  (e.g. every 500 ms).

These two rates can differ – OPC UA takes care of buffering and combining updates.


In [ ]:
class TemperatureSubHandler:
    """Callback handler that reacts to data changes and events."""

    def datachange_notification(self, node, val, data):
        ts = datetime.now().strftime("%H:%M:%S")
        print(f"[SUBSCRIBER {ts}] DataChange: Node={node}, Value={val}")

    def event_notification(self, event):
        ts = datetime.now().strftime("%H:%M:%S")
        print(f"[SUBSCRIBER {ts}] Event: {event}")

async def subscriber_client(runtime_seconds: float = 30.0, publishing_interval_ms: int = 500):
    """Subscribe to M01.Temperature and print all changes.

    :param runtime_seconds: How long the subscriber should stay active.
    :param publishing_interval_ms: OPC UA subscription publishing interval in milliseconds.
                                   This defines how often the server sends publish responses,
                                   not how often the value is written by the publisher client.
    """
    handler = TemperatureSubHandler()

    async with Client(url=SERVER_URL) as client:
        print("[SUBSCRIBER] Connected to server:", SERVER_URL)
        temp_node = await get_machine_temperature_node(client, "M01")

        # create_subscription returns a Subscription object in asyncua
        subscription = await client.create_subscription(publishing_interval_ms, handler)

        # subscribe_data_change returns a handle – useful if you later want to unsubscribe a single node
        handle = await subscription.subscribe_data_change(temp_node)
        print(f"[SUBSCRIBER] Subscription active (handle={handle}) for about {runtime_seconds} seconds ...")

        try:
            await asyncio.sleep(runtime_seconds)
        finally:
            print("[SUBSCRIBER] Deleting subscription ...")
            await subscription.delete()
            print("[SUBSCRIBER] Finished.")

## 6. Demo: run publisher and subscriber in parallel

The function below starts both clients in parallel:

- the **publisher** writes for a limited amount of time,  
- the **subscriber** stays active a bit longer,  
- both are awaited together.

> **Important:** Make sure your server notebook is already running before executing this cell.


In [ ]:
async def run_pub_sub_demo():
    """Run publisher and subscriber concurrently against the running factory server."""
    pub_task = asyncio.create_task(
        publisher_client(runtime_seconds=20.0, interval=1.0)
    )
    sub_task = asyncio.create_task(
        subscriber_client(runtime_seconds=30.0, publishing_interval_ms=500)
    )

    try:
        await asyncio.gather(pub_task, sub_task)
    except asyncio.CancelledError:
        print("[MAIN] Tasks have been cancelled.")


# Run the demo directly from the notebook
await run_pub_sub_demo()

## 7. Possible extensions

- Control several machines (`M01`–`M05`) in parallel and subscribe to each of them.
- Add more variables per machine (e.g. `State`, `Busy`) and include them in the clients.
